# meta-openenv — T4 Colab training (SFT + GRPO)

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. The setup cell clones `VanshGupta18/corporate-compliance-env` by default; change `GITHUB_USER` only if you are using a fork.
3. Run cells in order.

No WebSocket server is required; rollouts use in-process `ComplianceEnv`.

In [ ]:
print("Skipping direct TRL upgrade. Use the install cell below with pinned requirements.")


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

GITHUB_USER = "VanshGupta18"  # change if you are using a fork
REPO = "corporate-compliance-env"
BRANCH = "main"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO}.git"
WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORKDIR / REPO

if GITHUB_USER == "YOUR_GITHUB_USER":
    raise ValueError("Set GITHUB_USER to your GitHub username before running.")

WORKDIR.mkdir(parents=True, exist_ok=True)
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
elif REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 1) Install training dependencies

The `cuda-bindings` pip warning on Colab is harmless. This uses the Unsloth-compatible stack from `training/requirements-training.txt` and removes optional `torchcodec`, `torchao`, and `sentence-transformers` packages (they can break imports on Colab for text-only training). **After this cell finishes, use Runtime → Restart session, then rerun the clone/setup cell and verify cell.**

In [ ]:
import sys

print("Python:", sys.version)

# Core runtime deps
%pip install -q -r requirements.txt

# Colab T4 training stack (Unsloth-compatible pins)
%pip install -q -r training/requirements-training.txt

# Text-only training does not need these optional packages. On Colab they can
# break Unsloth imports via sentence-transformers / Transformers optional paths.
%pip uninstall -y -q torchcodec torchao sentence-transformers

print("Install complete.")
print("Next: Runtime → Restart session, then rerun clone/setup + verify.")

In [ ]:
# Import unsloth FIRST so it can patch the HF stack before transformers/trl load.
import unsloth  # noqa: F401

import importlib
import os
import sys
from pathlib import Path

import torch

REPO_DIR = Path("/content/corporate-compliance-env") if Path("/content").exists() else Path.cwd()
if not (REPO_DIR / "training" / "training_utils.py").exists():
    raise RuntimeError("Run the clone/setup cell first after restart.")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("cwd:", os.getcwd())
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Switch Colab runtime to T4 GPU before training.")
print("GPU:", torch.cuda.get_device_name(0))

for module in ("torch", "unsloth", "trl", "transformers", "peft", "accelerate", "datasets"):
    pkg = importlib.import_module(module)
    print(f"{module}:", getattr(pkg, "__version__", "ok"))

from training.training_utils import grpo_supports_rollout_func, native_grpo_supports_rollout_func

if not grpo_supports_rollout_func():
    raise RuntimeError("GRPO rollout_func support unavailable. Re-run install cell and restart runtime.")
print("TRL rollout_func:", "native" if native_grpo_supports_rollout_func() else "compat")
print("Stack ready — continue with dataset prep / dry runs.")

## 2) Optional: regenerate claims if splits are missing

In [ ]:
import pathlib
import subprocess
import sys

splits = list(pathlib.Path("data/splits").glob("*.json"))
if len(splits) < 3:
    subprocess.run(
        [
            sys.executable,
            "data/generate_dataset.py",
            "--train-per-diff",
            "120",
            "--val-per-diff",
            "40",
            "--test-per-diff",
            "40",
            "--seed",
            "42",
        ],
        check=True,
    )
else:
    print("data/splits present, skipping generate_dataset")

## 3) Prepare SFT data + dry runs

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT)

commands = [
    [sys.executable, "-m", "training.prepare_data", "--episodes-per-task", "40", "--split", "train"],
    [sys.executable, "-m", "training.sft_train", "--dry-run"],
    [sys.executable, "-m", "training.grpo_train", "--dry-run", "--curriculum-stage", "stage_1_easy"],
    [sys.executable, "-m", "training.smoke_test"],
]

for command in commands:
    print("$", " ".join(command))
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        env=env,
        capture_output=True,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout.rstrip())
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr.rstrip())
        completed.check_returncode()

### Reminder

If you skipped the post-install runtime restart, do **Runtime → Restart session** now, then rerun the clone/setup cell and verify cell before training.

In [ ]:
# Install order matters on Colab: requirements.txt → training/requirements-training.txt (see cell above).

## 4) (Optional) SFT warm start

GRPO can train directly from the base model (next section). Run this cell only if you want to warm-start with supervised fine-tuning first. Skip on T4 to save time.

In [ ]:
# Skip this cell unless you specifically want SFT warm-start. GRPO works directly from base model.
# !python -m training.sft_train \
#   --model-id unsloth/Qwen2.5-3B-Instruct-bnb-4bit \
#   --dataset-path training/data/sft_dataset.jsonl \
#   --output-dir training/checkpoints/sft \
#   --max-length 512 \
#   --batch-size 1 \
#   --grad-accum 8
print("Skipping SFT — GRPO will train from the base model in section 5.")

In [ ]:
# Keep TRL pinned to the Unsloth-compatible range from training/requirements-training.txt.
# If dependencies drifted mid-session, rerun the install cell and restart runtime.
print("TRL pin: from training/requirements-training.txt")

## 5) GRPO curriculum (stages 1 → 2 → 3)

Trains GRPO **directly from the base model** (no SFT required). Each stage continues from the previous adapter.

T4-safe defaults below use `--num-generations 1` and `--max-prompt-length 512`.

In [ ]:
!python -m training.grpo_train \
  --model-id unsloth/Qwen2.5-3B-Instruct-bnb-4bit \
  --curriculum-stage stage_1_easy \
  --output-dir training/checkpoints/grpo_stage1 \
  --max-seq-length 512 \
  --max-prompt-length 512 \
  --max-completion-length 96 \
  --num-generations 1 \
  --batch-size 1 \
  --grad-accum 8 \
  --max-train-steps 60 \
  --warmup-steps 10

In [ ]:
!python -m training.grpo_train \
  --sft-checkpoint training/checkpoints/grpo_stage1 \
  --curriculum-stage stage_2_medium \
  --output-dir training/checkpoints/grpo_stage2 \
  --max-seq-length 512 \
  --max-prompt-length 512 \
  --max-completion-length 96 \
  --num-generations 1 \
  --batch-size 1 \
  --grad-accum 8 \
  --max-train-steps 80 \
  --warmup-steps 10

In [ ]:
!python -m training.grpo_train \
  --sft-checkpoint training/checkpoints/grpo_stage2 \
  --curriculum-stage stage_3_hard \
  --output-dir training/checkpoints/grpo \
  --max-seq-length 512 \
  --max-prompt-length 512 \
  --max-completion-length 96 \
  --num-generations 1 \
  --batch-size 1 \
  --grad-accum 8 \
  --max-train-steps 120 \
  --warmup-steps 10

## 6) Evaluate checkpoint (in-process env)

In [ ]:
!python -m training.eval_checkpoint \
  --checkpoint training/checkpoints/grpo \
  --split validation \
  --episodes 10 \
  --episode-log-file training/logs/episodes.jsonl \
  --clear-log

## 7) Optional: publish adapter to Hugging Face

Create a model repo on the Hub, then set `HF_REPO_ID` and run the next cell.

In [ ]:
import sys

HF_REPO_ID = "YOUR_HF_USERNAME/compliance-grpo-adapter"  # edit

if HF_REPO_ID.startswith("YOUR_"):
    print("Set HF_REPO_ID to publish; skipping.")
else:
    import subprocess

    from huggingface_hub import notebook_login

    notebook_login()
    subprocess.run(
        [
            sys.executable,
            "-m",
            "training.publish_adapter",
            "--checkpoint",
            "training/checkpoints/grpo",
            "--repo-id",
            HF_REPO_ID,
        ],
        check=True,
    )